# UNIT IV — NLP Practice Programs

| # | Program | Library / Model | Technique |
|---|---------|----------------|-----------|
| 1 | Text Classification using RNN | PyTorch | Recurrent Neural Network |
| 2 | Sentiment Analysis using LSTM | PyTorch | Long Short-Term Memory |
| 3 | Text Classification using BERT | Transformers | Fine-tuned BERT |
| 4 | Sentiment Analysis using RoBERTa | Transformers | Fine-tuned RoBERTa |

## 📦 Installation
Run this cell once before executing any program.

In [1]:
!pip install torch transformers datasets scikit-learn -q

## Program 1: Text Classification using RNN

**Aim:** Build a Recurrent Neural Network (RNN) from scratch using PyTorch to classify text sequences.

**Algorithm:**
1. Prepare and tokenize a small text dataset.
2. Build vocabulary and convert words to integer indices.
3. Define an RNN model with Embedding → RNN → Linear layers.
4. Train the model using CrossEntropyLoss and Adam optimizer.
5. Evaluate the model on test sentences.

In [2]:
# Program 1: Text Classification using RNN
import torch
import torch.nn as nn
import numpy as np

# Sample dataset
sentences = [
    ("i love this movie", 1),
    ("this film is great", 1),
    ("wonderful performance", 1),
    ("i enjoyed watching it", 1),
    ("terrible movie ever", 0),
    ("waste of time", 0),
    ("very boring film", 0),
    ("i hated this movie", 0),
]

# Build vocabulary
all_words = set()
for sent, _ in sentences:
    all_words.update(sent.split())
vocab = {word: idx+1 for idx, word in enumerate(all_words)}
vocab['<PAD>'] = 0

def encode(sentence, max_len=6):
    tokens = [vocab.get(w, 0) for w in sentence.split()]
    tokens = tokens[:max_len] + [0] * (max_len - len(tokens))
    return tokens

X = torch.tensor([encode(s) for s, _ in sentences], dtype=torch.long)
y = torch.tensor([label for _, label in sentences], dtype=torch.long)

# RNN Model
class RNNClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, output_dim):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.rnn = nn.RNN(embed_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, output_dim)

    def forward(self, x):
        embedded = self.embedding(x)
        output, hidden = self.rnn(embedded)
        return self.fc(hidden.squeeze(0))

model = RNNClassifier(vocab_size=len(vocab)+1, embed_dim=16, hidden_dim=32, output_dim=2)
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
criterion = nn.CrossEntropyLoss()

# Training
print("=== RNN Training ===")
for epoch in range(30):
    model.train()
    optimizer.zero_grad()
    output = model(X)
    loss = criterion(output, y)
    loss.backward()
    optimizer.step()
    if (epoch+1) % 10 == 0:
        print(f"Epoch {epoch+1}/30 | Loss: {loss.item():.4f}")

# Testing
model.eval()
test_sentences = ["i love this film", "terrible and boring", "great wonderful movie"]
print("\n=== RNN Predictions ===")
for sent in test_sentences:
    x = torch.tensor([encode(sent)], dtype=torch.long)
    with torch.no_grad():
        out = model(x)
        pred = torch.argmax(out, dim=1).item()
    label = "Positive 😊" if pred == 1 else "Negative 😞"
    print(f"'{sent}' → {label}")

=== RNN Training ===
Epoch 10/30 | Loss: 0.0059
Epoch 20/30 | Loss: 0.0002
Epoch 30/30 | Loss: 0.0001

=== RNN Predictions ===
'i love this film' → Positive 😊
'terrible and boring' → Negative 😞
'great wonderful movie' → Positive 😊


## Program 2: Sentiment Analysis using LSTM

**Aim:** Implement an LSTM network using PyTorch to perform sentiment analysis on text data.

**Algorithm:**
1. Prepare tokenized text data with positive/negative labels.
2. Build vocabulary and encode sentences as padded integer sequences.
3. Define LSTM model with Embedding → LSTM → Linear layers.
4. Train using Binary Cross-Entropy loss.
5. Predict sentiment for new sentences.

In [3]:
# Program 2: Sentiment Analysis using LSTM
import torch
import torch.nn as nn

# Dataset
data = [
    ("the movie was fantastic and thrilling", 1),
    ("great acting and wonderful story", 1),
    ("i really enjoyed this film", 1),
    ("best movie i have seen", 1),
    ("absolutely loved every scene", 1),
    ("the movie was dull and boring", 0),
    ("terrible acting poor story", 0),
    ("i did not enjoy this at all", 0),
    ("worst film ever made", 0),
    ("complete waste of my time", 0),
]

# Vocabulary
all_words = set()
for sent, _ in data:
    all_words.update(sent.split())
vocab = {w: i+1 for i, w in enumerate(all_words)}
vocab['<PAD>'] = 0

def encode(sentence, max_len=8):
    tokens = [vocab.get(w, 0) for w in sentence.split()]
    return tokens[:max_len] + [0] * (max_len - len(tokens))

X = torch.tensor([encode(s) for s, _ in data], dtype=torch.long)
y = torch.tensor([[label] for _, label in data], dtype=torch.float)

# LSTM Model
class LSTMClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, 1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        embedded = self.embedding(x)
        _, (hidden, _) = self.lstm(embedded)
        return self.sigmoid(self.fc(hidden.squeeze(0)))

model = LSTMClassifier(vocab_size=len(vocab)+1, embed_dim=16, hidden_dim=32)
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
criterion = nn.BCELoss()

# Training
print("=== LSTM Training ===")
for epoch in range(40):
    model.train()
    optimizer.zero_grad()
    output = model(X)
    loss = criterion(output, y)
    loss.backward()
    optimizer.step()
    if (epoch+1) % 10 == 0:
        acc = ((output > 0.5).float() == y).float().mean().item()
        print(f"Epoch {epoch+1}/40 | Loss: {loss.item():.4f} | Accuracy: {acc*100:.1f}%")

# Testing
model.eval()
test_sentences = [
    "this was an amazing experience",
    "i hated every moment of it",
    "brilliant performances throughout",
    "boring and poorly made"
]
print("\n=== LSTM Predictions ===")
for sent in test_sentences:
    x = torch.tensor([encode(sent)], dtype=torch.long)
    with torch.no_grad():
        prob = model(x).item()
    label = "Positive 😊" if prob > 0.5 else "Negative 😞"
    print(f"'{sent}' → {label} (confidence: {prob:.2f})")

=== LSTM Training ===
Epoch 10/40 | Loss: 0.1766 | Accuracy: 100.0%
Epoch 20/40 | Loss: 0.0077 | Accuracy: 100.0%
Epoch 30/40 | Loss: 0.0017 | Accuracy: 100.0%
Epoch 40/40 | Loss: 0.0009 | Accuracy: 100.0%

=== LSTM Predictions ===
'this was an amazing experience' → Negative 😞 (confidence: 0.00)
'i hated every moment of it' → Positive 😊 (confidence: 1.00)
'brilliant performances throughout' → Negative 😞 (confidence: 0.00)
'boring and poorly made' → Negative 😞 (confidence: 0.00)


## Program 3: Text Classification using BERT

**Aim:** Use a pre-trained BERT model from HuggingFace Transformers to classify text sentiment.

**Algorithm:**
1. Load `bert-base-uncased` tokenizer and model with sequence classification head.
2. Tokenize input sentences with padding and attention masks.
3. Pass tokens through BERT to get logits.
4. Apply softmax to get class probabilities.
5. Output predicted sentiment label.

In [4]:
# Program 3: Text Classification using BERT
from transformers import BertTokenizer, BertForSequenceClassification
import torch
import torch.nn.functional as F

print("Loading BERT model... (this may take a minute)")
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
model = BertForSequenceClassification.from_pretrained(
    'bert-base-uncased',
    num_labels=2
)
model.eval()

# Labels: 0 = Negative, 1 = Positive
# Note: weights are random (not fine-tuned), so we demonstrate the pipeline
test_sentences = [
    "I absolutely loved this movie, it was fantastic!",
    "This was the worst film I have ever seen.",
    "The acting was brilliant and the story was captivating.",
    "Terrible plot and very boring throughout.",
    "An enjoyable and heartwarming experience.",
]

print("\n=== BERT Text Classification ===\n")
print(f"{'Sentence':<50} {'Prediction'}")
print("-" * 65)

for sentence in test_sentences:
    inputs = tokenizer(
        sentence,
        return_tensors='pt',
        truncation=True,
        padding=True,
        max_length=128
    )
    with torch.no_grad():
        outputs = model(**inputs)
        probs = F.softmax(outputs.logits, dim=1)
        pred = torch.argmax(probs, dim=1).item()
        confidence = probs[0][pred].item()

    label = "Positive 😊" if pred == 1 else "Negative 😞"
    short = sentence[:47] + "..." if len(sentence) > 50 else sentence
    print(f"{short:<50} {label} ({confidence:.2f})")

print("\nNote: Model uses pre-trained BERT weights.")
print("Fine-tuning on labeled data would improve accuracy significantly.")

Loading BERT model... (this may take a minute)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



=== BERT Text Classification ===

Sentence                                           Prediction
-----------------------------------------------------------------
I absolutely loved this movie, it was fantastic!   Positive 😊 (0.60)
This was the worst film I have ever seen.          Positive 😊 (0.61)
The acting was brilliant and the story was capt... Positive 😊 (0.61)
Terrible plot and very boring throughout.          Positive 😊 (0.60)
An enjoyable and heartwarming experience.          Positive 😊 (0.60)

Note: Model uses pre-trained BERT weights.
Fine-tuning on labeled data would improve accuracy significantly.


## Program 4: Sentiment Analysis using RoBERTa

**Aim:** Use a fine-tuned RoBERTa model to perform accurate sentiment analysis on input text.

**Algorithm:**
1. Load `cardiffnlp/twitter-roberta-base-sentiment` — a RoBERTa model fine-tuned on sentiment data.
2. Tokenize input text using the RoBERTa tokenizer.
3. Run a forward pass to get logits.
4. Apply softmax and map output to Negative / Neutral / Positive labels.
5. Display results with confidence scores.

In [5]:
# Program 4: Sentiment Analysis using RoBERTa
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch
import torch.nn.functional as F

print("Loading RoBERTa model... (fine-tuned on sentiment data)")
model_name = "cardiffnlp/twitter-roberta-base-sentiment"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)
model.eval()

labels = ['Negative 😞', 'Neutral 😐', 'Positive 😊']

test_sentences = [
    "I love this product, it works perfectly!",
    "The weather today is okay I guess.",
    "This is absolutely terrible, I want a refund.",
    "Just received my order. It came on time.",
    "Best purchase I have ever made, highly recommend!",
    "Not sure how I feel about this.",
    "Complete disaster, nothing worked at all.",
]

print("\n=== RoBERTa Sentiment Analysis ===\n")
print(f"{'Sentence':<48} {'Sentiment':<18} {'Confidence'}")
print("-" * 78)

for sentence in test_sentences:
    inputs = tokenizer(
        sentence,
        return_tensors='pt',
        truncation=True,
        padding=True,
        max_length=128
    )
    with torch.no_grad():
        outputs = model(**inputs)
        probs = F.softmax(outputs.logits, dim=1)
        pred = torch.argmax(probs, dim=1).item()
        confidence = probs[0][pred].item()

    short = sentence[:45] + "..." if len(sentence) > 48 else sentence
    print(f"{short:<48} {labels[pred]:<18} {confidence*100:.1f}%")

print("\n✅ RoBERTa is fine-tuned — these predictions are accurate!")

Loading RoBERTa model... (fine-tuned on sentiment data)


config.json:   0%|          | 0.00/747 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/150 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/499M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-roberta-base-sentiment
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



=== RoBERTa Sentiment Analysis ===

Sentence                                         Sentiment          Confidence
------------------------------------------------------------------------------
I love this product, it works perfectly!         Positive 😊         99.2%
The weather today is okay I guess.               Positive 😊         93.2%
This is absolutely terrible, I want a refund.    Negative 😞         97.7%
Just received my order. It came on time.         Positive 😊         84.0%
Best purchase I have ever made, highly recomm... Positive 😊         98.3%
Not sure how I feel about this.                  Negative 😞         60.1%
Complete disaster, nothing worked at all.        Negative 😞         94.5%

✅ RoBERTa is fine-tuned — these predictions are accurate!
